In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

### That alter dataset showed coupling I really want to see if it works with the unseen pairs.

In [30]:
import os
import torch
from utils.load_pipeline import GenerateEvulatePairs
from torch.utils.data import DataLoader, random_split
from script.model_dim_4_layer_3 import FibonacciModDatasetSmallShample, model, MinimalTransformer, evaluate_model, train_model, train_plot, eval_plot


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



checkpoint_dir = '../checkpoints'
full_path = os.path.join(checkpoint_dir, 'dim_4_layer_3_alterset.pth')

vocab_size = 10
model = MinimalTransformer(vocab_size=vocab_size).to(device=device)

model.load_state_dict(
    torch.load(full_path, map_location=device)['model_state_dict']
)

batch_size = 8

generated_ds = FibonacciModDatasetSmallShample(mod=vocab_size, seq_len=20)

train_size = int(0.8 * len(generated_ds))
test_size = len(generated_ds) - train_size
train_ds, test_ds = random_split(generated_ds, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)

avg_loss, eval_accuracy  = evaluate_model(model, test_loader, show_accuracy=True)

unseen = GenerateEvulatePairs(dataset=train_ds, mod=vocab_size)
if len(unseen) == 0:
    print("No unseen data!")



Accuracy (eval mode): 99.07%
100
No unseen data!


In [ ]:
import random
vocab_size = 10
batch_size = 16
seq_len = 20


all_seeds = [(i, j) for i in range(vocab_size) for j in range(vocab_size)]
random.shuffle(all_seeds)

split = int(0.6 * len(all_seeds))
train_seeds = all_seeds[:split]
test_seeds  = all_seeds[split:]


train_ds = FibonacciModDatasetSmallShample(
    num_samples=25000,
    mod=vocab_size,
    seq_len=seq_len,
    seeds=train_seeds
)

test_ds = FibonacciModDatasetSmallShample(
    num_samples=5000,
    mod=vocab_size,
    seq_len=seq_len,
    seeds=test_seeds
)


train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size)

epoch = 22
total_accuray = 0 

model = MinimalTransformer(vocab_size=vocab_size).to(device)
file_name = "dim_4_layer_3_alterset.pth"

train_model(model, train_loader, epochs=epoch, test_loader=test_loader)
avg_loss, eval_accuracy  = evaluate_model(model, test_loader, show_accuracy=True)

if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

checkpoint = {
    'model_state_dict': model.state_dict(),
    'train_loss_history': train_plot,
    'eval_loss_history': eval_plot,
    'epoch': epoch,
    'total_accuracy': eval_accuracy, 
}
torch.save(checkpoint, full_path)
print(f"Successfully saved to: {full_path}")


Epoch 1, Loss (Training): 1.4778 Loss(val): 2.3853
Epoch 2, Loss (Training): 1.1445 Loss(val): 2.5368
Epoch 3, Loss (Training): 1.0189 Loss(val): 2.6762
Epoch 4, Loss (Training): 0.9608 Loss(val): 2.8642
Epoch 5, Loss (Training): 0.9413 Loss(val): 3.0184
Epoch 6, Loss (Training): 0.9134 Loss(val): 3.1485
Epoch 7, Loss (Training): 0.9103 Loss(val): 3.1016
Epoch 8, Loss (Training): 0.8927 Loss(val): 3.1850
Epoch 9, Loss (Training): 0.8816 Loss(val): 3.1845
Epoch 10, Loss (Training): 0.8825 Loss(val): 3.0934
Epoch 11, Loss (Training): 0.8638 Loss(val): 3.1056
Epoch 12, Loss (Training): 0.8610 Loss(val): 3.2679
Epoch 13, Loss (Training): 0.8542 Loss(val): 3.2805
Epoch 14, Loss (Training): 0.8435 Loss(val): 3.3838
Epoch 15, Loss (Training): 0.8419 Loss(val): 3.2985
Epoch 16, Loss (Training): 0.8456 Loss(val): 3.2252
Epoch 17, Loss (Training): 0.8417 Loss(val): 3.2830
Epoch 18, Loss (Training): 0.8373 Loss(val): 3.3781
Epoch 19, Loss (Training): 0.8304 Loss(val): 3.2926
Epoch 20, Loss (Train